# FocusTrack Component 1 - Exploratory Analysis

Context-Aware Learner Interaction State Modeling.

This notebook reads the artifacts written by `scripts/run_pipeline.py` and
explores feature distributions, discovered interaction states, transition
structure, context differences and the baseline/proposed/ablation comparison.

Run the pipeline first:

```bash
python scripts/run_pipeline.py --source csv --window 60 --states 4
```

In [1]:
from pathlib import Path

import pandas as pd

from src.config import FEATURES_DIR, RESULTS_DIR
from src.visualization.plots import (
    plot_experiment_summary,
    plot_feature_distributions,
    plot_state_profiles,
    plot_transition_heatmap,
)

FIGURES = RESULTS_DIR / "figures"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60)

## 1. Load the window-level feature table

In [2]:
features = pd.read_csv(FEATURES_DIR / "features.csv", parse_dates=["window_start", "window_end"])
features.shape

(360, 47)

In [3]:
features.head()

,participant_id,session_id,context,window_size_seconds,window_index,window_start,window_end,session_progress,idle_ratio,mean_idle_time,n_snapshots,key_event_count,typing_duration,typing_rate,mean_keyboard_count,std_keyboard_count,max_keyboard_count,burst_count,mean_burst_duration,mean_burst_length,mouse_event_count,mouse_distance,mouse_clicks,click_count,mean_mouse_velocity,std_mouse_velocity,mean_mouse_acceleration,mouse_click_rate,application_duration,application_switch_count,unique_application_count,application_switch_frequency,current_application_category,application_sequence,focus_reference,interaction_state,previous_state,transition_type,is_transition,state_duration,stable_state_duration,state_change_rate,state_entropy,interruption_count,recovery_count,context_code,context_state_id
0,00000001-0000-0000-0000-000000000000,00000001-0000-0000-0000-000000000000,assignment,60.0,0,2026-09-01 08:00:00+00:00,2026-09-01 08:00:59+00:00,0.000000,0.0,3.046317,60,49.0,32.0,0.816667,0.816667,1.008161,4.0,16,2.000000,2.000000,60,2587.046329,47.0,47.0,43.117439,26.846278,30.007830,0.783333,60.0,2,3,0.033333,terminal,browser|communication|terminal,NaN,1,NaN,start,False,1,1,0.000000,-0.000000,0,0,4,13.0
1,00000001-0000-0000-0000-000000000000,00000001-0000-0000-0000-000000000000,assignment,60.0,1,2026-09-01 08:01:00+00:00,2026-09-01 08:01:59+00:00,0.100167,0.0,2.986251,60,49.0,38.0,0.816667,0.816667,0.718602,2.0,14,2.714286,2.714286,60,3085.084759,49.0,49.0,51.418079,38.267836,45.548526,0.816667,60.0,0,1,0.000000,terminal,terminal,NaN,2,1.0,1->2,True,1,1,0.500000,0.693147,1,0,4,14.0
2,00000001-0000-0000-0000-000000000000,00000001-0000-0000-0000-000000000000,assignment,60.0,2,2026-09-01 08:02:00+00:00,2026-09-01 08:02:59+00:00,0.200334,0.0,3.035235,60,52.0,35.0,0.866667,0.866667,0.884433,3.0,12,2.916667,2.916667,60,3525.051025,43.0,43.0,58.750850,41.066761,47.288617,0.716667,60.0,1,2,0.016667,code_editor,terminal|code_editor,NaN,2,2.0,start,False,2,2,0.333333,0.636514,1,0,4,14.0
3,00000001-0000-0000-0000-000000000000,00000001-0000-0000-0000-000000000000,assignment,60.0,3,2026-09-01 08:03:00+00:00,2026-09-01 08:03:59+00:00,0.300501,0.0,3.301603,60,64.0,39.0,1.066667,1.066667,1.030642,4.0,16,2.437500,2.437500,60,3405.912959,50.0,50.0,56.765216,42.710422,48.072206,0.833333,60.0,1,2,0.016667,code_editor,code_editor|browser,NaN,2,2.0,start,False,3,3,0.250000,0.562335,1,0,4,14.0
4,00000001-0000-0000-0000-000000000000,00000001-0000-0000-0000-000000000000,assignment,60.0,4,2026-09-01 08:04:00+00:00,2026-09-01 08:04:59+00:00,0.400668,0.0,2.940334,60,57.0,33.0,0.950000,0.950000,1.023474,4.0,17,1.941176,1.941176,60,2488.177786,49.0,49.0,41.469630,35.710149,44.082364,0.816667,60.0,1,2,0.016667,browser,browser|office,2.0,2,2.0,start,False,4,4,0.200000,0.500402,1,0,4,14.0


## 2. Learning-context coverage

How many windows and sessions fall into each learning activity?

In [4]:
features.groupby("context").agg(
    windows=("window_index", "size"),
    sessions=("session_id", "nunique"),
    participants=("participant_id", "nunique"),
)

,windows,sessions,participants
context,,,
assignment,60,6,6
coding,120,12,9
reading,80,8,7
writing,40,4,3
zoom,60,6,5


## 3. Feature distributions

In [5]:
plot_feature_distributions(
    features,
    ["typing_rate", "mouse_distance", "idle_ratio", "application_switch_frequency"],
    FIGURES / "feature_distributions.png",
)

PosixPath('/home/Dhananjana/Studies/Research/J26-DS-307/FocusTrack_Component1/data/results/figures/feature_distributions.png')

## 4. Discovered interaction states

In [6]:
profile = pd.read_csv(FEATURES_DIR / "state_profiles.csv")
plot_state_profiles(profile, FIGURES / "state_profiles.png")
profile

,state,n_windows,typing_rate,typing_rate_level,mean_mouse_velocity,mean_mouse_velocity_level,idle_ratio,idle_ratio_level,application_switch_frequency,application_switch_frequency_level,mouse_clicks,mouse_clicks_level,common_context,mean_focus_reference
0,0,129,3.506460,High,45.956410,Low,0.0,Low,0.003101,Medium,23.581395,Low,coding,4.605634
1,1,86,0.760271,Low,50.814589,High,0.0,Low,0.042054,High,46.802326,High,reading,2.098039
2,2,145,0.764023,Low,49.814290,Medium,0.0,Low,0.009080,Medium,44.344828,Medium,reading,2.480000


## 5. State transitions

`P(next state | current state)`, overall and within each learning context.

In [7]:
transition_probs = pd.read_csv(RESULTS_DIR / "transition_probabilities.csv")
plot_transition_heatmap(transition_probs, FIGURES / "transition_heatmap.png")
transition_probs.head(10)

,from_state,to_state,count,probability
0,0,0,116,1.000000
1,1,1,34,0.459459
2,1,2,40,0.540541
3,2,0,1,0.007463
4,2,1,46,0.343284
5,2,2,87,0.649254


## 6. Does the meaning of a state depend on context?

In [8]:
context_analysis = pd.read_csv(RESULTS_DIR / "context_effect_analysis.csv")
context_agreement = pd.read_csv(RESULTS_DIR / "context_state_agreement.csv", index_col=0)
context_analysis

,context,n_windows,n_labeled_windows,mean_focus_reference,n_states,state_focus_spread,highest_focus_state,lowest_focus_state,dominant_state
0,assignment,60,33,3.060606,3,2.512987,0,1,2
1,coding,120,66,3.833333,3,2.556818,0,1,0
2,reading,80,44,2.522727,2,0.557143,2,1,2
3,writing,40,22,3.818182,3,1.812500,0,1,0
4,zoom,60,32,2.218750,2,0.748918,2,1,2


In [9]:
context_agreement

,coding,reading,writing,zoom
assignment,,,,
1.000000,1.000000,1.0,0.866025,1.0
1.000000,1.000000,1.0,0.866025,1.0
1.000000,1.000000,1.0,NaN,1.0
0.866025,0.866025,NaN,1.000000,NaN
1.000000,1.000000,1.0,NaN,1.0


## 7. Baseline vs proposed vs ablation

In [10]:
summary = pd.read_csv(RESULTS_DIR / "experiment_summary.csv")
plot_experiment_summary(summary, FIGURES / "experiment_summary.png")
summary

,representation,model,split,n_folds,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,accuracy_mean,recall_macro_mean,precision_macro_mean,roc_auc_mean
0,baseline,xgboost,cross_context,5,0.414816,0.132932,0.489113,0.516863,0.440919,0.433206,0.674062
1,ablation_no_temporal,xgboost,cross_context,5,0.406947,0.151941,0.479947,0.521886,0.432363,0.468122,0.736539
2,ablation_no_temporal,random_forest,cross_context,5,0.379991,0.083064,0.456554,0.483430,0.424173,0.428333,0.799486
3,ablation_no_transition,xgboost,cross_context,5,0.372174,0.135298,0.488124,0.488617,0.455743,0.418127,0.704741
4,baseline,random_forest,cross_context,5,0.362322,0.118345,0.465855,0.449744,0.435855,0.395557,0.802224
5,ablation_no_state,random_forest,cross_context,5,0.362062,0.083945,0.466656,0.482522,0.466656,0.368253,0.840727
6,proposed,random_forest,cross_context,5,0.355827,0.068899,0.472607,0.474593,0.442607,0.366905,0.753820
7,ablation_no_context,xgboost,cross_context,5,0.349432,0.097719,0.456041,0.463617,0.431160,0.364873,0.700412
8,ablation_no_context,random_forest,cross_context,5,0.342847,0.099456,0.462190,0.465907,0.462190,0.334021,0.780362
9,proposed,xgboost,cross_context,5,0.335180,0.106447,0.430186,0.447617,0.405305,0.353476,0.707124


## 8. Sequential (HMM) view of interaction states

Hidden behavioural regimes inferred from the sequences of interaction states,
and whether hidden-Markov sequence features add predictive information.

In [11]:
hidden_transitions = pd.read_csv(RESULTS_DIR / "hmm_hidden_transitions.csv", index_col=0)
hidden_transitions

,hidden_0,hidden_1,hidden_2
from_hidden,,,
hidden_0,1.000000e+00,3.283488e-25,4.007069e-25
hidden_1,1.330708e-13,8.997762e-01,1.002238e-01
hidden_2,1.386532e-02,3.390241e-01,6.471106e-01


In [12]:
hmm_session = pd.read_csv(RESULTS_DIR / "hmm_session_features.csv")
hmm_session.head()

,participant_id,session_id,hmm__log_likelihood,hmm__mean_dwell,hmm__transition_entropy,hmm__dominant,hmm__share_0,hmm__share_1,hmm__share_2
0,00000001-0000-0000-0000-000000000000,00000001-0000-0000-0000-000000000000,-0.787572,3.333407e+11,-0.000000,1,0.0,1.0,0.0
1,00000001-0000-0000-0000-000000000000,00000002-0000-0000-0000-000000000000,-0.803330,3.333407e+11,-0.000000,1,0.0,1.0,0.0
2,00000001-0000-0000-0000-000000000000,00000003-0000-0000-0000-000000000000,-0.689098,3.333407e+11,0.325083,1,0.0,0.9,0.1
3,00000002-0000-0000-0000-000000000000,00000004-0000-0000-0000-000000000000,-0.423571,3.333407e+11,-0.000000,2,0.0,0.0,1.0
4,00000002-0000-0000-0000-000000000000,00000005-0000-0000-0000-000000000000,-0.732863,3.333407e+11,-0.000000,1,0.0,1.0,0.0


In [13]:
sequential_summary = pd.read_csv(RESULTS_DIR / "sequential_summary.csv")
sequential_summary

,representation,model,split,n_folds,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,accuracy_mean,recall_macro_mean,precision_macro_mean,roc_auc_mean
0,proposed,svm,participant,5,0.534143,0.126008,0.568524,0.63,0.547286,0.545131,0.644651
1,proposed_hmm,svm,participant,5,0.476267,0.064942,0.536024,0.58,0.489786,0.497845,0.641068
2,proposed,random_forest,participant,5,0.443110,0.090289,0.501429,0.56,0.482524,0.467222,0.870525
3,proposed_hmm,random_forest,participant,5,0.439777,0.084313,0.491429,0.56,0.472524,0.470794,0.857689
4,proposed_hmm,xgboost,participant,5,0.406502,0.093544,0.465143,0.53,0.427238,0.429841,0.764892
5,proposed,xgboost,participant,5,0.368228,0.090849,0.409429,0.48,0.392524,0.384603,0.745585


## 9. Statistical analysis

Paired comparisons against the baseline (positive `mean_delta` favours the
target), plus variation between participants and between learning contexts.

In [14]:
statistics = pd.read_csv(RESULTS_DIR / "statistics_comparison.csv")
statistics

,n,mean_reference,mean_target,mean_delta,median_delta,ci_low,ci_high,p_wilcoxon,p_ttest,cohens_dz,cliffs_delta,reference,target,model,split,metric
0,5,0.442287,0.409310,-0.032977,-0.037607,-0.071694,0.003744,0.3125,0.224171,-0.642494,-0.12,baseline,ablation_no_context,random_forest,participant,macro_f1
1,5,0.442287,0.427800,-0.014486,-0.020766,-0.071033,0.052359,0.4375,0.693379,-0.189615,-0.20,baseline,ablation_no_state,random_forest,participant,macro_f1
2,5,0.442287,0.514437,0.072151,0.033654,-0.009178,0.153479,0.3750,0.210200,0.666837,0.40,baseline,ablation_no_temporal,random_forest,participant,macro_f1
3,5,0.442287,0.423555,-0.018731,-0.004748,-0.049573,0.006974,0.4375,0.302931,-0.528280,-0.12,baseline,ablation_no_transition,random_forest,participant,macro_f1
4,5,0.442287,0.443110,0.000823,0.000000,-0.041491,0.039783,1.0000,0.974434,0.015248,0.00,baseline,proposed,random_forest,participant,macro_f1
5,5,0.410076,0.424133,0.014056,-0.021429,-0.029810,0.066645,0.8125,0.636433,0.228408,-0.28,baseline,ablation_no_context,svm,participant,macro_f1
6,5,0.410076,0.520511,0.110434,0.067727,0.004403,0.229270,0.1875,0.157219,0.777224,0.68,baseline,ablation_no_state,svm,participant,macro_f1
7,5,0.410076,0.535745,0.125669,0.136757,0.041264,0.213607,0.1250,0.059317,1.167963,0.44,baseline,ablation_no_temporal,svm,participant,macro_f1
8,5,0.410076,0.453447,0.043370,0.048185,-0.020385,0.105429,0.3125,0.293449,0.540407,0.28,baseline,ablation_no_transition,svm,participant,macro_f1
9,5,0.410076,0.534143,0.124066,0.134935,0.018225,0.229908,0.3125,0.116439,0.893341,0.52,baseline,proposed,svm,participant,macro_f1


In [15]:
variation = pd.read_csv(RESULTS_DIR / "statistics_variation.csv")
participant_variation = pd.read_csv(RESULTS_DIR / "participant_variation.csv")
context_variation = pd.read_csv(RESULTS_DIR / "context_variation.csv")
variation

,group,n_groups,mean,std,min,max
0,participant,25,0.414889,0.251814,0.125000,1.000000
1,context,5,0.363289,0.035997,0.333333,0.418421


In [16]:
participant_variation

,participant_id,n_sessions,macro_f1,balanced_accuracy
0,00000001-0000-0000-0000-000000000000,4,0.166667,0.250000
1,00000002-0000-0000-0000-000000000000,4,0.200000,0.333333
2,00000003-0000-0000-0000-000000000000,4,0.200000,0.333333
3,00000004-0000-0000-0000-000000000000,4,0.333333,0.500000
4,00000005-0000-0000-0000-000000000000,4,0.333333,0.500000
5,00000006-0000-0000-0000-000000000000,4,0.200000,0.333333
6,00000007-0000-0000-0000-000000000000,4,0.375000,0.666667
7,00000008-0000-0000-0000-000000000000,4,0.333333,0.500000
8,00000009-0000-0000-0000-000000000000,4,1.000000,1.000000
9,00000010-0000-0000-0000-000000000000,4,0.555556,0.750000
